## 7. N-HiTS Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
    !pip install -q pytorch_forecasting
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

IN_COLAB=False  REPO_ROOT=/Users/ucabtt4/Projects/Dissertation/dissertation


In [2]:
import json

from src.dataloader import load_cgm_stats, load_tensors
from src import metrics
from src import train
from src.models.nn_baseline import NHiTSPredictor, HORIZONS

MODEL_NAME = 'NHiTS'
SEED = 42  # change this and re-run for a multi-seed stability comparison
# Built from REPO_ROOT, not the current working directory: Colab's cwd is /content regardless of
# where this notebook file itself lives on the mounted Drive, so a plain relative path like
# Path('results') would silently write to the wrong (ephemeral) place.
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_PATH = EXPERIMENT_DIR / 'results' / f'nhits_seed{SEED}_results.json'
CHECKPOINT_DIR = EXPERIMENT_DIR / 'checkpoints' / f'nhits_seed{SEED}'

In [4]:
stats = load_cgm_stats()
train_loader = load_tensors('train')  # batch_size=256 default (Training Procedure)
val_loader = load_tensors('val')
test_loader = load_tensors('test')

print('n_train_batches', len(train_loader), 'n_val_batches', len(val_loader),
      'n_test_batches', len(test_loader))

sample_cgm, sample_target = next(iter(train_loader))
print('input batch shape (cgm):', tuple(sample_cgm.shape), sample_cgm.dtype)
print('target batch shape:', tuple(sample_target.shape), sample_target.dtype)

n_train_batches 327 n_val_batches 66 n_test_batches 66
input batch shape (cgm): (256, 24) torch.float32
target batch shape: (256, 5) torch.float32


In [5]:
train.set_seed(SEED)  # before constructing the model -- weight init draws from this
model = NHiTSPredictor(context_length=sample_cgm.shape[1])  # tie to actual data shape, not a
                                                              # hardcoded 24, in case the window
                                                              # size ever changes upstream

model, history = train.train_model(
    model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
    checkpoint_dir=CHECKPOINT_DIR,
)

device: cpu  model: NHiTSPredictor  params: 70,002
epoch   1/50  train_loss=0.3987  val_loss=0.3374  val_rmse(mg/dL)=[11.5, 18.3, 25.3, 29.5, 31.9]
epoch   2/50  train_loss=0.3818  val_loss=0.3339  val_rmse(mg/dL)=[11.2, 17.8, 25.2, 29.4, 31.9]
epoch   3/50  train_loss=0.3783  val_loss=0.3374  val_rmse(mg/dL)=[11.2, 18.0, 25.3, 29.6, 32.1]
epoch   4/50  train_loss=0.3755  val_loss=0.3378  val_rmse(mg/dL)=[11.3, 17.9, 25.3, 29.6, 32.1]
epoch   5/50  train_loss=0.3741  val_loss=0.3337  val_rmse(mg/dL)=[11.1, 17.7, 25.2, 29.4, 32.0]
epoch   6/50  train_loss=0.3726  val_loss=0.3320  val_rmse(mg/dL)=[11.1, 17.7, 25.1, 29.4, 31.9]
epoch   7/50  train_loss=0.3719  val_loss=0.3341  val_rmse(mg/dL)=[11.0, 17.6, 25.1, 29.5, 32.1]
epoch   8/50  train_loss=0.3705  val_loss=0.3372  val_rmse(mg/dL)=[11.3, 18.1, 25.3, 29.5, 32.0]
epoch   9/50  train_loss=0.3701  val_loss=0.3348  val_rmse(mg/dL)=[11.2, 17.7, 25.2, 29.5, 32.0]
epoch  10/50  train_loss=0.3693  val_loss=0.3316  val_rmse(mg/dL)=[11.0, 17.

In [6]:
y_pred_val, y_true_val = train.predict(model, val_loader)
y_pred_test, y_true_test = train.predict(model, test_loader)

val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])

print('=== Validation (best checkpoint) ===')
print(metrics.format_results_table(val_results))
print()
print('=== Test ===')
print(metrics.format_results_table(test_results))

=== Validation (best checkpoint) ===
Horizon (min) |     RMSE |      MAE |    gRMSE | Zone A+B %
-----------------------------------------------------------
           15 |    10.80 |     7.23 |    11.72 |      99.71
           30 |    17.38 |    11.51 |    19.57 |      99.34
           60 |    24.96 |    16.59 |    29.51 |      98.64
           90 |    29.31 |    19.76 |    35.33 |      98.16
          120 |    31.89 |    21.88 |    38.79 |      97.97

=== Test ===
Horizon (min) |     RMSE |      MAE |    gRMSE | Zone A+B %
-----------------------------------------------------------
           15 |    11.05 |     7.42 |    12.17 |      99.87
           30 |    18.31 |    12.06 |    21.13 |      99.60
           60 |    27.04 |    17.83 |    32.84 |      99.14
           90 |    32.30 |    21.62 |    40.06 |      98.61
          120 |    35.31 |    23.87 |    44.24 |      98.25


In [7]:
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seed': SEED,
        'val_results': val_results,
        'test_results': test_results,
        'n_epochs_trained': len(history['train_loss']),
    }, f, indent=2)

print(f'saved to {RESULTS_PATH}')
print(f'checkpoints (best.pt / final.pt / history.json) saved to {CHECKPOINT_DIR}')

saved to /Users/ucabtt4/Projects/Dissertation/dissertation/experiments/1_baseline/results/nhits_seed42_results.json
checkpoints (best.pt / final.pt / history.json) saved to /Users/ucabtt4/Projects/Dissertation/dissertation/experiments/1_baseline/checkpoints/nhits_seed42
